In [0]:
from pyspark.sql.functions import col, expr, trim

# -----------------------------
# 1. Read data
# -----------------------------
df = spark.read.table("mini_project_catalog.`01_bronze`.estimate")

# -----------------------------
# 2. Remove duplicates
# -----------------------------
df = df.dropDuplicates()

# -----------------------------
# 3. Replace invalid values ("-") with null
# -----------------------------
df = df.replace("-", None)

# -----------------------------
# 4. Trim all string columns
# -----------------------------
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# -----------------------------
# 5. Convert timestamp column (ISO format)
# -----------------------------
if "created_at" in df.columns:
    df = df.withColumn("created_at", expr("to_timestamp(created_at)"))

# -----------------------------
# 6. Convert numeric columns
# -----------------------------
if "estimate_amount" in df.columns:
    df = df.withColumn(
        "estimate_amount",
        expr("CAST(try_cast(estimate_amount AS DOUBLE) AS DOUBLE)")
    )

if "version_no" in df.columns:
    df = df.withColumn("version_no", col("version_no").cast("int"))

# -----------------------------
# 7. Filter valid records
# -----------------------------
df = df.filter(col("estimate_id").isNotNull())
df = df.filter(col("order_id").isNotNull())

# -----------------------------
# 8. Standardize text columns
# -----------------------------
df = df.withColumn("estimate_type", col("estimate_type"))

# -----------------------------
# 9. (IMPORTANT 🔥) Keep latest version per order
# -----------------------------
# Business logic: latest estimate = highest version_no

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("order_id").orderBy(col("version_no").desc())

df = df.withColumn("rn", row_number().over(window_spec)) \
       .filter(col("rn") == 1) \
       .drop("rn")

# -----------------------------
# 10. Write to Silver
# -----------------------------
df.write.mode("overwrite").saveAsTable(
    "mini_project_catalog.02_silver.estimate"
)

In [0]:
%sql
SELECT * FROM mini_project_catalog.02_silver.estimate LIMIT 10;